# Segmentation and response calculation

A **segmenter** defines units as character spans in the stimulus text. A response's
**native units** are the source authors' measurement areas, represented by its
annotation layer. These can be words, multiword regions or characters; they need
not match whitespace boundaries or model tokens.

Changing predictor units does not create finer response measurements. Subword
eye-tracking responses require fixation locations precise enough to assign those
subwords: character landings, or gaze coordinates mapped to characters using the
actual display layout and an independently validated alignment. Coordinates alone
are insufficient. The shared event mapper accepts the resulting character offsets
or source-area assignments; it does not infer a display layout from raw coordinates.

The code cells below run offline. Examples that require corpus files appear in
Markdown code blocks. We cover segmenters, additive responses, fixation
recalculation, skip policy, evaluation, and EEG/fMRI timing.

In [ ]:
import numpy as np

from replica.core import AnnotationLayer, Datum, Span, Stimulus
from replica.segmentation import (
    CharacterSegmenter,
    WhitespaceSegmenter,
    aggregate_mask,
    aggregate_values,
    compute_refinement,
    from_predicate,
    segment_stimulus,
)

## 1. Built-in Segmenters

A `Segmenter` is any callable that maps `text -> list[Span]`. Each `Span` is a half-open character interval `[start, end)` into the raw text.

In [ ]:
text = "The cat sat on the mat."

# WhitespaceSegmenter: splits on whitespace boundaries
word_seg = WhitespaceSegmenter()
word_spans = word_seg(text)

print(f"Text: {text!r}\n")
print(f"WhitespaceSegmenter (name={word_seg.name!r}):")
for _i, span in enumerate(word_spans):
    print(f"  [{span.start:2d}:{span.end:2d})  {text[span.start : span.end]!r}")

In [ ]:
# CharacterSegmenter: each character is a unit (whitespace skipped by default)
char_seg = CharacterSegmenter()
char_spans = char_seg(text)

print(f"CharacterSegmenter (name={char_seg.name!r}, include_whitespace=False):")
print(f"  {len(char_spans)} spans")
for span in char_spans[:10]:
    print(f"  [{span.start:2d}:{span.end:2d})  {text[span.start : span.end]!r}", end="")
print("  ...")

# Including whitespace
char_seg_ws = CharacterSegmenter(include_whitespace=True)
char_spans_ws = char_seg_ws(text)
print("\nCharacterSegmenter (include_whitespace=True):")
print(f"  {len(char_spans_ws)} spans (= len(text) = {len(text)})")

## 2. Custom Segmenters from Boundary Predicates

`from_predicate(pred, name)` builds a segmenter from a function that identifies **separator** characters. Characters where the predicate returns `True` are excluded from all spans.

In [ ]:
# Split on hyphens
hyphen_seg = from_predicate(lambda t, i: t[i] == "-", "morphemes")
hyphenated = "well-known-fact"
spans = hyphen_seg(hyphenated)

print(f"Text: {hyphenated!r}")
print(f"Segmenter: {hyphen_seg.name!r} (separator = '-')")
for span in spans:
    print(f"  [{span.start}:{span.end})  {hyphenated[span.start : span.end]!r}")

In [ ]:
# Split on whitespace and punctuation -> "tokens"
import string

token_seg = from_predicate(
    lambda t, i: t[i].isspace() or t[i] in string.punctuation,
    "tokens",
)

text_with_punct = "Hello, world! How's it going?"
spans = token_seg(text_with_punct)

print(f"Text: {text_with_punct!r}")
print(f"Segmenter: {token_seg.name!r} (separator = whitespace + punctuation)")
for span in spans:
    print(
        f"  [{span.start:2d}:{span.end:2d})  {text_with_punct[span.start : span.end]!r}"
    )

In [ ]:
# "Sentence" segmenter: predicate that never fires -> entire text is one unit
sentence_seg = from_predicate(lambda _t, _i: False, "sentences")
spans = sentence_seg(text)

print(f"Text: {text!r}")
print(f"Segmenter: {sentence_seg.name!r} (no separator -> one span)")
for span in spans:
    print(f"  [{span.start}:{span.end})  {text[span.start : span.end]!r}")

## 3. Applying Segmenters to Stimuli

`segment_stimulus()` applies a segmenter and registers the result as an annotation layer on the stimulus. If an identical layer already exists, it's reused.

In [ ]:
# Build a synthetic stimulus whose native units are whitespace-separated
text = "The cat sat on the mat"
words = text.split()
spans = []
pos = 0
for w in words:
    spans.append(Span(pos, pos + len(w)))
    pos += len(w) + 1
stim = Stimulus(text=text, stimulus_id="example")
stim.add_layer(AnnotationLayer("words", spans, words))

print(f"Before segmentation: layers = {list(stim.layers.keys())}")

# WhitespaceSegmenter matches the existing "words" layer -> reused
layer_name = segment_stimulus(stim, WhitespaceSegmenter())
print(f"\nWhitespaceSegmenter -> layer {layer_name!r} (reused existing)")
print(f"  Layers now: {list(stim.layers.keys())}")

# CharacterSegmenter adds a new "characters" layer
layer_name = segment_stimulus(stim, CharacterSegmenter())
print(f"\nCharacterSegmenter -> layer {layer_name!r} (new)")
print(f"  Layers now: {list(stim.layers.keys())}")
print(f"  Characters: {stim.layers['characters'].extract_strings(text)}")

## 4. Span Refinement and Compatibility

Evaluating data recorded at one granularity (e.g. words) at another (e.g. sentences) requires a **refinement** check: do the data spans cleanly nest inside the target spans?

`compute_refinement(source_spans, target_spans)` checks this and returns a mapping from target units to the source units they contain.

In [ ]:
# Example: words refine a sentence (coarser target)
word_spans = [
    Span(0, 3),
    Span(4, 7),
    Span(8, 11),
    Span(12, 14),
    Span(15, 18),
    Span(19, 22),
]
sentence_span = [Span(0, 22)]  # entire text as one unit

mapping = compute_refinement(word_spans, sentence_span)

print("Words -> Sentence refinement:")
print(f"  Word spans:     {[(s.start, s.end) for s in word_spans]}")
print(f"  Sentence span:  {[(s.start, s.end) for s in sentence_span]}")
print(f"  Mapping:        sentence[0] contains word indices {mapping[0]}")
print(f"  = {[text[word_spans[i].start : word_spans[i].end] for i in mapping[0]]}")

In [ ]:
# Characters refine words
text_short = "ab cd"
char_spans = [Span(0, 1), Span(1, 2), Span(3, 4), Span(4, 5)]  # a, b, c, d (skip space)
word_spans_short = [Span(0, 2), Span(3, 5)]  # "ab", "cd"

mapping = compute_refinement(char_spans, word_spans_short)

print("Characters -> Words refinement:")
for t_idx, s_indices in enumerate(mapping):
    word = text_short[word_spans_short[t_idx].start : word_spans_short[t_idx].end]
    chars = [text_short[char_spans[i].start : char_spans[i].end] for i in s_indices]
    print(f"  word[{t_idx}] {word!r} <- chars {chars}")

In [ ]:
# Incompatible: word spans don't refine character spans (can't split words into chars)
print("Incompatible case: trying to map word spans into character target spans...")
try:
    compute_refinement(word_spans_short, char_spans)
except ValueError as e:
    print(f"  ValueError: {e}")

## 5. Aggregating Additive Values

`aggregate_values()` sums arrays whose values are additive over disjoint source units. The example below uses synthetic SPR times. It does not calculate fixation windows or check source assignment rules.

Eye-tracking responses use fixation events when the loader supports them: events are mapped to target spans before calculating first fixation, first run, go-past and total time. First-fixation, first-run and go-past summaries must never be summed.

In [ ]:
# Simulate word-level SPR reading times for "The cat sat on the mat"
word_rts = np.array([120.0, 250.0, 180.0, 90.0, 110.0, 200.0])
words = ["The", "cat", "sat", "on", "the", "mat"]

# Target: two phrases -- "The cat sat" and "on the mat"
phrase_spans = [Span(0, 11), Span(12, 22)]
word_spans = [
    Span(0, 3),
    Span(4, 7),
    Span(8, 11),
    Span(12, 14),
    Span(15, 18),
    Span(19, 22),
]

mapping = compute_refinement(word_spans, phrase_spans)

# Aggregate by summation
phrase_rts = aggregate_values(word_rts, mapping)

print("Word-level RTs:")
for w, rt in zip(words, word_rts, strict=False):
    print(f"  {w:6s}  {rt:6.0f} ms")

print("\nPhrase-level RTs (summed):")
print(
    f"  'The cat sat'  -> {phrase_rts[0]:.0f} ms  (= {' + '.join(str(int(word_rts[i])) for i in mapping[0])})"
)
print(
    f"  'on the mat'   -> {phrase_rts[1]:.0f} ms  (= {' + '.join(str(int(word_rts[i])) for i in mapping[1])})"
)

In [ ]:
# NaN propagation: if any word in a phrase is missing, the phrase value is NaN
word_rts_with_nan = np.array([120.0, np.nan, 180.0, 90.0, 110.0, 200.0])
phrase_rts_nan = aggregate_values(word_rts_with_nan, mapping)

print("With missing data (word 'cat' = NaN):")
print(f"  'The cat sat'  -> {phrase_rts_nan[0]}  (NaN because 'cat' is missing)")
print(f"  'on the mat'   -> {phrase_rts_nan[1]:.0f} ms  (all present)")

# Mask aggregation: a phrase is included only if all its words are
word_mask = np.array([True, False, True, True, True, True])
phrase_mask = aggregate_mask(word_mask, mapping)
print(f"\nWord mask:   {word_mask}")
print(f"Phrase mask: {phrase_mask}  (first phrase excluded because 'cat' is excluded)")

## 6. Response Recalculation

`additive_measures` declares which summaries permit summation when no event adapter is available. It is not a list of all measures that support custom units. With an event adapter, `evaluate()` first validates native responses and recomputes the requested measures from mapped fixations. A source disagreement raises instead of falling back to summation.

An explicit `FixationCalculation` always requires events, even for an additive measure. Without events, first fixation, first run and go-past cannot be reaggregated.

In [ ]:
from replica.dataset import EyeTrackingDataset, SPRDataset

print("SPRDataset.additive_measures:")
print(f"  {SPRDataset.additive_measures}")

print("\nEyeTrackingDataset.additive_measures:")
print(f"  {EyeTrackingDataset.additive_measures}")

print("\nWhich eye-tracking measures can be aggregated by summation?")
for m in [
    "total_time",
    "total_fixation_duration",
    "first_run",
    "first_fixation",
    "go_past",
]:
    additive = m in EyeTrackingDataset.additive_measures
    print(f"  {m:30s}  {'additive' if additive else 'NOT additive'}")

### Recalculate a response from a dataset

This example needs Provo's summary and fixation files:

```python
from replica.dataset import load_dataset

provo = load_dataset("provo")
native = provo.data("first_fixation")[0]
passage = from_predicate(lambda text, index: False, "passage")
layer = segment_stimulus(native.stimulus, passage)
response = provo.reaggregate(native, layer)
print(response.values)
```

Before calculating custom units, `reaggregate()` performs a **native round trip**:
it maps the source events back to their original measurement areas, recomputes the
native measure, and compares every response and its missingness with the selected native reference
array. The absolute tolerance is `1e-6` ms, with no mismatch allowance. This check
precedes skip masking. It establishes that the event selection and spatial mapping
recover the reference; it does not establish that every alternative measure has
an independently released reference.

The mapper then assigns physical events to the target spans and recalculates the
reading windows. Merging words changes which departures are regressions and which
fixations belong to one visit. First fixation remains one event's duration;
first-fixation, first-run and go-past summaries must never be summed.

`EvaluationSpec(segmentation=...)` uses this same response path during evaluation.
Spans must be ordered, disjoint and within the source text. Evaluation rejects
ambiguous sentence assignments, including regions spanning source sentences.

In Dundee, uncertain source positions use `candidate_unit_indices`, with no
assigned word or character position. A `None` candidate includes an off-text
landing. A target region resolves an event only if all candidates map to that same
region; otherwise affected windows remain unavailable. Merging candidate areas
can resolve uncertainty, but cannot establish a subword landing.

## 7. Integration with `evaluate()`

`EvaluationSpec.segmentation=None` uses the response's native layer. A segmenter
creates a target layer and checks response compatibility automatically.

This synthetic example uses one reader, OLS and stimulus folds to demonstrate
alignment. The predictor is unit length; the random responses do not establish a
scientific effect. A character segmenter cannot split the recorded word times.

In [ ]:
from replica import (
    OLS,
    AnalysisConfig,
    EvaluationSpec,
    ItemSplitter,
    UnitLength,
    evaluate,
)
from replica.dataset import DatasetInfo, SPRDataset


# --- Build a small toy dataset ---
class ToyDataset(SPRDataset):
    info = DatasetInfo(name="toy")

    def load(self, path):
        pass


def make_toy():
    ds = ToyDataset()
    rng = np.random.default_rng(42)
    for i in range(8):
        words = ["a", "short", "synthetic", "sentence", "for", "alignment"]
        text = " ".join(words)
        spans, pos = [], 0
        for w in words:
            spans.append(Span(pos, pos + len(w)))
            pos += len(w) + 1
        stim = Stimulus(text=text, stimulus_id=f"s{i}")
        stim.add_layer(AnnotationLayer("words", spans, words))
        ds._stimuli.append(stim)
        ds._data.append(
            Datum(
                stimulus=stim,
                measure_name="spr_rt",
                values=rng.uniform(200, 600, size=6),
                participant_id="p0",
            )
        )
    return ds


ds = make_toy()
config = AnalysisConfig(model=OLS(), design=None, cv=ItemSplitter(3))

In [ ]:
# Default: no segmentation -> uses the data's native "words" layer
features = EvaluationSpec(
    predictors=[UnitLength()],
    targets=[ds.feature("spr_rt")],
)
results = evaluate(ds, features, config)
print("Default (no segmenter):")
print(f"  delta-LL per unit: {results[0].delta_ll_per_unit:+.6f}")
print(f"  n_observations:    {results[0].n_observations}")

In [ ]:
# With WhitespaceSegmenter: matches existing "words" layer -> same result
features_ws = EvaluationSpec(
    predictors=[UnitLength()],
    targets=[ds.feature("spr_rt")],
    segmentation=WhitespaceSegmenter(),
)
results_ws = evaluate(ds, features_ws, config)
print("WhitespaceSegmenter (matches native layer):")
print(f"  delta-LL per unit: {results_ws[0].delta_ll_per_unit:+.6f}")
print(f"  n_observations:    {results_ws[0].n_observations}")

In [ ]:
# This SPR source has word times and no finer response measurements
print("CharacterSegmenter on word-level data:")
try:
    features_char = EvaluationSpec(
        predictors=[UnitLength()],
        targets=[ds.feature("spr_rt")],
        segmentation=CharacterSegmenter(),
    )
    evaluate(ds, features_char, config)
except ValueError as e:
    print(f"  ValueError: {e}")

## 8. Writing Your Own Segmenter

Subclass `Segmenter` and implement the `segment` method. This makes it straightforward to wrap a regex, an FST, or any other segmentation algorithm.

In [ ]:
import re

from replica.segmentation import Segmenter


class RegexSegmenter(Segmenter):
    """Segment text using a regex pattern to find units."""

    def __init__(self, pattern: str, name: str):
        super().__init__(name)
        self._pattern = re.compile(pattern)

    def segment(self, text: str) -> list[Span]:
        return [Span(m.start(), m.end()) for m in self._pattern.finditer(text)]


# Example: segment into sentences (split on ". " or end-of-string)
sentence_seg = RegexSegmenter(r"[^.]+\.?", "sentences")
multi_sentence = "The cat sat. The dog ran. The bird flew."
spans = sentence_seg(multi_sentence)

print(f"Text: {multi_sentence!r}\n")
print("RegexSegmenter (pattern=sentence-like):")
for _i, span in enumerate(spans):
    print(
        f"  [{span.start:2d}:{span.end:2d})  {multi_sentence[span.start : span.end]!r}"
    )

# Verify it's a proper Segmenter
print(f"\nisinstance(sentence_seg, Segmenter): {isinstance(sentence_seg, Segmenter)}")

## 9. Compatibility summary

| Available response information | Target units | Handling |
|---|---|---|
| Native summaries | Same spans | Read the selected source; explicit calculations require events |
| Fixation events with source-area assignments | Whole or merged source areas | Validate native measures, map events, recompute windows |
| Fixation events with character landings | Finer spans supported by those landings | Validate native measures, map events, recompute windows |
| Additive summaries without an event adapter | Complete source units combined into larger spans | Sum only where the loader declares additivity |
| Non-additive summaries without events | Different spans | Reject |
| Word-area assignments without finer positions | Subword spans | Reject |

Inspect `describe_dataset(name).fixations` for the registered event capability and
the loader's source requirements. The [eye-tracking guide](../datasets/eye_tracking.ipynb)
generates the current dataset inventory. A registered capability does not mean the
required source files have been downloaded.

## Skip policy

Eye-tracking loaders default to `drop_skips=True`: observed zero durations are
excluded from duration analyses. This targets duration conditional on fixation.
Including zeros mixes predicting whether a reader fixates with predicting how
long they fixate; a gain from predicting skips should not be interpreted as a gain
on positive reading times alone.

Use `load_dataset("provo", drop_skips=False)` to retain observed zeros. Missing
source observations remain missing. The model and response transformation must
also admit zero: taking its logarithm will not retain it as a finite response.
[Pimentel et al. (2023), §4.2](https://doi.org/10.1162/tacl_a_00603) compare policies;
[Wilcox et al. (2023), §3.1](https://aclanthology.org/2023.tacl-1.82/)
include first-pass skips as zero. State the policy with each result and compare
models on the same eligible observations.

## Sources, measures and calculations

The measure, source and calculation are separate choices. `source="released"`
selects the retained summary, with the loader's unit conversion and missingness.
An explicit calculation uses that source as its native validation reference and
recomputes events on the requested units. It requires matching event files.

```python
from replica.dataset import FixationCalculation, fixation_feature, load_dataset

meco = load_dataset("meco", release="l1_w1", language="en")
standard = meco.data("first_run")
released = meco.data("first_run", source="released")
calculation = FixationCalculation(first_run="popeye_episode@1")
response = fixation_feature(
    meco, "first_run", source="released", calculation=calculation,
)
# Use response in EvaluationSpec.targets; retain the explicit calculation policy.
assert FixationCalculation.from_dict(calculation.to_dict()) == calculation
```

MECO defaults to contiguous first runs; its released episode-based durations can
include a departure and return. CopCo defaults to standard go-past; its released
extractor can omit parts of the regression path. Source selections retain those
distinctions. Generic ambiguous labels such as `gaze_rt` are rejected; use a named
measure from the dataset catalogue and an explicit source.

| Measure | Window |
|---|---|
| `first_fixation` | First fixation on the unit, whenever reached |
| `first_run` | Consecutive fixations from first arrival to departure |
| `go_past` | First arrival until rightward exit, including regressions |
| `right_bounded` | On-unit fixations before the first rightward landing |
| `total_time` | All on-unit fixations across the selected presentations |
| `first_pass` | Whether first arrival precedes every landing to the right |

The `_in_first_pass` duration variants require first-pass entry. Known first-pass
skips are zero before masking; unknown status remains unknown. Source policies
determine presentation boundaries, off-text dwell, rounding and required exits.

`FixationCalculation` selects versioned alternatives to `standard@1`:

| Field | Alternative | Meaning |
|---|---|---|
| `first_run` | `popeye_episode@1` | On-unit dwell in the first regression episode reaching it |
| `go_past` | `copco_extractor@1` | CopCo's event-index-adjacent subset of the path |
| `first_run`, `go_past`, `total_time` | `modelblocks@1` | ModelBlocks windows with recorded interruption intervals |

Version numbers identify REPLICA definitions. Upstream revisions are recorded
separately in the returned response provenance. Native run labels cannot substitute
for rebuilding episodes on custom spans. The [pinned popEye source archive](https://github.com/sascha2schroeder/popEye/archive/93ce662b571eda30162d24e2da34433c1a146e77.zip)
contains `R/ComputeRegression.R`, `R/ComputeRun.R` and
`R/AggregateWordsFirstrun.R` at the retained revision. The archive link avoids the
unavailable GitHub file pages for that revision.
The other implementations are the pinned
[CopCo extractor](https://github.com/norahollenstein/copco-processing/blob/0b3bd294a3c09e186c551c085b709423181947f9/extract_features.py)
and [ModelBlocks extractor](https://github.com/modelblocks/modelblocks-release/blob/3ae5120327005cde1c02576eda24876997e20550/resource-dundee/scripts/process_dundee.py).

For a measure absent from the summary, use
`dataset.reaggregate(native, layer, measure="right_bounded")`. The native anchor
must still match. An unreferenced quantity or explicit alternative records
`has_native_reference=False`; the anchor check does not prove numerical agreement
for the new quantity. `AnalysisResult.response` records the resolved source, calculation,
skip policy and target fingerprint. Different calculations require distinct feature names and retain separate provenance.

## Source constraints

- Event access can require files beyond the summary: CELER needs `sent_fix.tsv`
  alongside `sent_ia.tsv`; MECO requires matching word and fixation report versions.
  Follow [CELER's acquisition instructions](https://github.com/berzak/celer#obtaining).
- ZuCo stores one physical event per fixation. Source selection applies its author
  filter, including the 100 ms threshold. Shared-boundary events can belong to both
  released word summaries; custom units count physical events once and preserve
  unresolved candidate assignments. Released totals are therefore not additive.
  Synthetic cumulative dwell timestamps do not establish presentation timing.
- MECO preserves blank source areas for path calculations. The L2 wave 2 `ch_s`
  release has no go-past reference and uses contiguous released first runs.
- Dundee ModelBlocks screen/line conflicts can leave custom-unit assignments
  unresolved. RSC's later fixation reports do not match its summary cohorts.
- GECO, RastrOS and de Varda inputs provide summaries without usable participant
  fixation streams. Only declared additive measures can be combined without events.

RSC's evaluator defaults to `exclude_inconsistent_go_past=True`, excluding finite
go-past durations below the same observation's released first-run duration. This
also applies to the first-pass restriction. `data()` retains source values; set
the option to `False` to retain them during evaluation. This consistency check does
not turn an excluded duration into a skip or reconcile the source reports.

## EEG and fMRI

Text segmentation and temporal alignment are separate operations. Word-aligned
ERP amplitudes and band-power summaries are not additive reading durations: a
segmenter does not justify summing them into phrase-level neural responses.
Finer temporal analyses require recordings, event timing and an explicitly chosen
response window or encoding model.

`dataset.recordings()` returns sampled `Recording` objects. Their first array axis
is time; remaining named axes can describe electrodes, bands or voxels. Timing is
stored as a sampling interval or explicit sample times. Recording-specific
`event_times`, where supplied, contain onsets for the stimulus's `words` layer;
they do not automatically acquire timings for newly segmented spans.

`evaluate()` with `EncodingConfig` dispatches to the continuous encoding path.
That path aligns event predictors to response sample times and models temporal
delays. Its alignment and delay settings must match the acquisition and scientific
question. See the [EEG](../datasets/eeg.ipynb) and [fMRI](../datasets/fmri.ipynb)
guides for the currently supported sources; fMRI support is preliminary.